In [ ]:
import ee
ee.Initialize(project="ee-mnolasco")

deptos = ee.FeatureCollection("users/mnolasco/FORMAR_2024_2026/departments_selected")
piloto = deptos.filter(ee.Filter.eq("departamen", "Río Segundo"))

print("Encontrados:", piloto.size().getInfo())
print(piloto.first().toDictionary(["departamen", "provincia", "link"]).getInfo())

In [ ]:
geom = piloto.geometry()

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(geom)
      .filterDate("2021-02-01", "2021-02-11"))

nubes = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
s2 = s2.linkCollection(nubes, ["cs_cdf"])

print("Escenas encontradas:", s2.size().getInfo())
fechas = s2.aggregate_array("system:time_start").getInfo()
print(sorted({ee.Date(f).format("YYYY-MM-dd").getInfo() for f in fechas}))

In [ ]:
def preparar(img):
    limpio = img.select("cs_cdf").gte(0.60)
    return img.select(["B2", "B3", "B4"]).divide(10000).toFloat().updateMask(limpio)

comp = s2.map(preparar).median()

gcc = comp.select("B3").divide(
    comp.select("B2").add(comp.select("B3")).add(comp.select("B4"))
).rename("gcc")

base = "users/mnolasco/FORMAR_2024_2026/"
agricola = ee.Image(base + "AgriLand-2019").eq(1)
soja_inta = ee.Image(base + "MNC_verano-2021").eq(11)

reductor = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)

for nombre, mascara in [("agrícola", agricola), ("soja INTA (control)", agricola.And(soja_inta))]:
    stats = gcc.updateMask(mascara).reduceRegion(
        reducer=reductor, geometry=geom, scale=60, maxPixels=1e10, bestEffort=True)
    print(nombre, stats.getInfo())

In [ ]:
import pandas as pd

def gcc_intervalo(fecha):
    ini = ee.Date(fecha)
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(geom).filterDate(ini, ini.advance(10, "day"))
           .linkCollection(nubes, ["cs_cdf"]).map(preparar))
    relleno = ee.Image.constant([0, 0, 0]).rename(["B2", "B3", "B4"]).toFloat().updateMask(0)
    m = col.merge(ee.ImageCollection([relleno])).median()
    g = m.select("B3").divide(m.select("B2").add(m.select("B3")).add(m.select("B4")))
    dos = g.updateMask(agricola).rename("agricola").addBands(
          g.updateMask(agricola.And(soja_inta)).rename("soja"))
    s = dos.reduceRegion(ee.Reducer.mean(), geom, 60, maxPixels=1e10, bestEffort=True)
    return ee.Feature(None, s.set("fecha", fecha).set("n_escenas", col.size()))

fechas = pd.date_range("2020-11-01", "2021-05-31", freq="10D").strftime("%Y-%m-%d").tolist()

filas = []
for i, f in enumerate(fechas, 1):
    props = gcc_intervalo(f).toDictionary().getInfo()
    filas.append(props)
    print(f"{i}/{len(fechas)}  {f}  escenas={props.get('n_escenas')}  "
          f"agricola={props.get('agricola')}  soja={props.get('soja')}")

serie = pd.DataFrame(filas).sort_values("fecha").reset_index(drop=True)
serie

In [ ]:
import matplotlib.pyplot as plt

serie["fecha"] = pd.to_datetime(serie["fecha"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(serie["fecha"], serie["agricola"], "o-", label="Área agrícola (método)")
ax.plot(serie["fecha"], serie["soja"], "s--", label="Soja INTA 2021 (control)")
ax.axvspan(pd.Timestamp("2021-02-09"), pd.Timestamp("2021-03-01"),
           color="gold", alpha=0.3, label="Ventana Chen et al. (DOY 40–60)")
ax.set_ylabel("GCC")
ax.set_title("Río Segundo — campaña 2020/21")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
serie.to_csv("../results/tables/serie_gcc_rio_segundo_2021.csv", index=False)

In [ ]:
import ee
from IPython.display import Image, display
ee.Initialize(project="ee-mnolasco")

deptos = ee.FeatureCollection("users/mnolasco/FORMAR_2024_2026/departments_selected")
geom = deptos.filter(ee.Filter.eq("link", "14119")).geometry()

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(geom)
      .filterDate("2021-02-09", "2021-03-11"))

fechas = sorted(set(s2.aggregate_array("system:time_start")
                      .map(lambda t: ee.Date(t).format("YYYY-MM-dd")).getInfo()))
print("Fechas con imagen:", fechas)

vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 2500,
       "region": geom.bounds(), "dimensions": 512, "format": "png"}

for f in fechas:
    d = ee.Date(f)
    img = s2.filterDate(d, d.advance(1, "day")).mosaic().clip(geom)
    print(f)
    display(Image(url=img.getThumbURL(vis)))

In [ ]:
import ee
from IPython.display import HTML, display
ee.Initialize(project="ee-mnolasco")

deptos = ee.FeatureCollection("users/mnolasco/FORMAR_2024_2026/departments_selected")
geom = deptos.filter(ee.Filter.eq("link", "14119")).geometry()

# Exactamente el intervalo de la fila "2021-02-19": del 19 al 28 de febrero
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(geom)
      .filterDate("2021-02-09", "2021-02-19")
      .linkCollection(ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED"), ["cs_cdf"]))

print("Escenas usadas:")
for nombre in s2.aggregate_array("system:index").getInfo():
    print("  ", nombre)

vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 2500,
       "region": geom.bounds(), "dimensions": 512, "format": "png"}

def lado_a_lado(img1, img2, titulo):
    u1, u2 = img1.getThumbURL(vis), img2.getThumbURL(vis)
    display(HTML(f"<b>{titulo}</b><br><img src='{u1}' width=420> <img src='{u2}' width=420>"))

fechas = sorted(set(s2.aggregate_array("system:time_start")
                      .map(lambda t: ee.Date(t).format("YYYY-MM-dd")).getInfo()))
for f in fechas:
    d = ee.Date(f)
    dia = s2.filterDate(d, d.advance(1, "day")).mosaic().clip(geom)
    usado = dia.updateMask(dia.select("cs_cdf").gte(0.60))
    lado_a_lado(dia, usado, f"{f} — izquierda: imagen completa · derecha: píxeles usados")

comp = (s2.map(lambda i: i.updateMask(i.select("cs_cdf").gte(0.60)))
          .median().clip(geom))
display(HTML("<b>Mediana del intervalo (lo que entró al cálculo del GCC)</b><br>"
             f"<img src='{comp.getThumbURL(vis)}' width=420>"))

In [ ]:
col = s2.map(lambda i: i.select("B3").updateMask(i.select("cs_cdf").gte(0.60)))
n = col.count()

dif = col.median().subtract(col.mean()).updateMask(n.eq(2))
print("Mediana − promedio, donde hay 2 valores:",
      dif.reduceRegion(ee.Reducer.minMax(), geom, 60, maxPixels=1e10, bestEffort=True).getInfo())

print("Cantidad de valores por píxel (n: píxeles):",
      n.reduceRegion(ee.Reducer.frequencyHistogram(), geom, 60, maxPixels=1e10, bestEffort=True).getInfo())

In [ ]:
base = "users/mnolasco/FORMAR_2024_2026/"
agricola = ee.Image(base + "AgriLand-2019").eq(1)
cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

todas = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(geom).filterDate("2021-02-09", "2021-03-11")
         .linkCollection(cs, ["cs_cdf"]))

fechas = sorted(set(todas.aggregate_array("system:time_start")
                         .map(lambda t: ee.Date(t).format("YYYY-MM-dd")).getInfo()))

def preparar(i):
    refl = i.select(["B2", "B3", "B4"]).divide(10000).toFloat()
    return refl.updateMask(i.select("cs_cdf").gte(0.60)).addBands(i.select("cs_cdf"))

total = agricola.selfMask().reduceRegion(ee.Reducer.count(), geom, 60,
                                         maxPixels=1e10, bestEffort=True).getInfo()
total = list(total.values())[0]

for f in fechas:
    d = ee.Date(f)
    dia = todas.filterDate(d, d.advance(1, "day")).map(preparar).mosaic()
    g = dia.select("B3").divide(dia.select("B2").add(dia.select("B3")).add(dia.select("B4")))
    img = g.rename("gcc").addBands(dia.select("cs_cdf").rename("cs")).updateMask(agricola)
    r = img.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
                         geom, 60, maxPixels=1e10, bestEffort=True).getInfo()
    print(f"{f}   GCC={r['gcc_mean']:.4f}   área con dato={100*r['gcc_count']/total:5.1f} %   "
          f"Cloud Score+ medio={r['cs_mean']:.2f}")

In [ ]:
import pandas as pd
import numpy as np

base = "users/mnolasco/FORMAR_2024_2026/"
agricola = ee.Image(base + "AgriLand-2019").eq(1)
soja_inta = ee.Image(base + "MNC_verano-2021").eq(11)
cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

def preparar(i):
    return (i.select(["B2", "B3", "B4"]).divide(10000).toFloat()
             .updateMask(i.select("cs_cdf").gte(0.60)))

def gcc_intervalo(fecha):
    ini = ee.Date(fecha)
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(geom).filterDate(ini, ini.advance(10, "day"))
           .linkCollection(cs, ["cs_cdf"]).map(preparar))
    relleno = ee.Image.constant([0, 0, 0]).rename(["B2", "B3", "B4"]).toFloat().updateMask(0)
    m = col.merge(ee.ImageCollection([relleno])).median()
    g = m.select("B3").divide(m.select("B2").add(m.select("B3")).add(m.select("B4")))
    dos = g.updateMask(agricola).rename("agricola").addBands(
          g.updateMask(agricola.And(soja_inta)).rename("soja"))
    s = dos.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
                         geom, 60, maxPixels=1e10, bestEffort=True)
    return ee.Feature(None, s.set("fecha", fecha).set("n_escenas", col.size()))

fechas = pd.date_range("2020-11-01", "2021-05-31", freq="10D").strftime("%Y-%m-%d").tolist()
filas = []
for i, f in enumerate(fechas, 1):
    p = gcc_intervalo(f).toDictionary().getInfo()
    p["cobertura"] = 100 * (p.get("agricola_count") or 0) / total
    filas.append(p)
    print(f"{i:2d}/{len(fechas)}  {f}  cobertura={p['cobertura']:5.1f} %  GCC={p.get('agricola_mean')}")

serie = pd.DataFrame(filas).rename(columns={"agricola_mean": "agricola", "soja_mean": "soja"})
serie = serie[["fecha", "n_escenas", "cobertura", "agricola", "soja"]].sort_values("fecha").reset_index(drop=True)

COBERTURA_MIN = 50
serie["valido"] = serie["cobertura"] >= COBERTURA_MIN
serie.to_csv("../results/tables/serie_gcc_rio_segundo_2021.csv", index=False)
serie

In [ ]:
S2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(geom).filterDate("2020-11-01", "2021-06-01")
      .linkCollection(cs, ["cs_cdf"]))

fechas_s2 = sorted(set(S2.aggregate_array("system:time_start")
                         .map(lambda t: ee.Date(t).format("YYYY-MM-dd")).getInfo()))
print(len(fechas_s2), "fechas con imagen\n")

def preparar_fecha(i):
    refl = i.select(["B2", "B3", "B4"]).divide(10000).toFloat()
    return refl.updateMask(i.select("cs_cdf").gte(0.60)).addBands(i.select("cs_cdf").rename("cs"))

reductor = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)

filas = []
for k, f in enumerate(fechas_s2, 1):
    d = ee.Date(f)
    dia = S2.filterDate(d, d.advance(1, "day")).map(preparar_fecha).mosaic()
    g = dia.select("B3").divide(dia.select("B2").add(dia.select("B3")).add(dia.select("B4")))
    img = (g.updateMask(agricola).rename("agricola")
            .addBands(g.updateMask(agricola.And(soja_inta)).rename("soja"))
            .addBands(dia.select("cs").updateMask(agricola)))
    r = img.reduceRegion(reductor, geom, 60, maxPixels=1e10, bestEffort=True).getInfo()
    cob = 100 * (r.get("agricola_count") or 0) / total
    filas.append({"fecha": f, "cobertura": cob, "cs_medio": r.get("cs_mean"),
                  "agricola": r.get("agricola_mean"), "soja": r.get("soja_mean")})
    print(f"{k:2d}/{len(fechas_s2)}  {f}  cobertura={cob:5.1f} %  "
          f"cs={r.get('cs_mean') or 0:.2f}  GCC={r.get('agricola_mean')}")

por_fecha = pd.DataFrame(filas)
por_fecha["valida"] = por_fecha["cobertura"] >= COBERTURA_MIN
por_fecha.to_csv("../results/tables/serie_gcc_por_fecha_rio_segundo_2021.csv", index=False)
por_fecha[por_fecha["valida"]]

In [ ]:
v = por_fecha[por_fecha["valida"]].copy()
v["fecha"] = pd.to_datetime(v["fecha"])
v = v.sort_values("fecha").reset_index(drop=True)

v["dias"] = v["fecha"].diff().dt.days
v["rho"] = v["agricola"].pct_change()
v["rho_dia"] = v["rho"] / v["dias"]

EPS = 0.005
v["estable_paper"] = v["rho"].abs() < EPS
v["estable_dia"] = v["rho_dia"].abs() < EPS / 5

v[["fecha", "dias", "agricola", "rho", "rho_dia", "estable_paper", "estable_dia"]].round(4)

In [ ]:
import matplotlib.pyplot as plt

v = por_fecha[por_fecha["valida"]].copy()
v["fecha"] = pd.to_datetime(v["fecha"])
v = v.sort_values("fecha").reset_index(drop=True)
v["rho"] = v["agricola"].pct_change()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for ax, eps in zip(axes, [0.005, 0.03]):
    ax.plot(v["fecha"], v["agricola"], "o-", color="0.4", label="GCC área agrícola")
    for i in v.index[1:]:
        if abs(v.loc[i, "rho"]) < eps:
            ax.axvspan(v.loc[i - 1, "fecha"], v.loc[i, "fecha"], color="gold", alpha=0.5)
    ax.axvspan(pd.Timestamp("2021-02-09"), pd.Timestamp("2021-03-01"),
               fill=False, hatch="//", edgecolor="tab:blue", label="Chen et al. (DOY 40–60)")
    ax.set_title(f"Tramos con |ρ| < {eps}  (amarillo)")
    ax.set_ylabel("GCC")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

def ventana_por_pico(serie, columna="agricola", campania=2021,
                     cobertura_min=50, umbral_pico=0.80, n_vecinas=2, k=2):
    """Ventana de máxima estabilidad del GCC alrededor del pico (enero-marzo)."""
    s = serie[serie["cobertura"] >= cobertura_min].copy()
    s["fecha"] = pd.to_datetime(s["fecha"])
    s = s.sort_values("fecha").reset_index(drop=True)

    # 1. Base (suelo desnudo) = mínimo GCC de la campaña; pico = máximo en ene-mar
    base = s[columna].min()
    em = s[(s["fecha"] >= f"{campania}-01-01") & (s["fecha"] < f"{campania}-04-01")].reset_index(drop=True)
    ip = em[columna].idxmax()
    pico = em.loc[ip, columna]
    em["pct_recorrido"] = 100 * (em[columna] - base) / (pico - base)

    # 2. Fechas de referencia: hasta n_vecinas antes y después del pico, con >= umbral_pico del recorrido
    ok = em["pct_recorrido"] >= 100 * umbral_pico
    antes = em.index[(em.index < ip) & ok][-n_vecinas:]
    despues = em.index[(em.index > ip) & ok][:n_vecinas]
    ref = sorted([*antes, ip, *despues])
    res = {"fecha_pico": em.loc[ip, "fecha"].date(), "gcc_pico": round(pico, 4), "gcc_base": round(base, 4),
           "fechas_ref": [em.loc[i, "fecha"].date().isoformat() for i in ref]}
    if len(ref) < 3:
        return {**res, "t1": None, "t2": None, "motivo": "menos de 3 fechas de referencia: revisar"}

    # 3. epsilon = k * desvío estándar de los rho entre fechas de referencia consecutivas
    rho_ref = em.loc[ref, columna].pct_change().dropna()
    eps = k * rho_ref.std(ddof=1)

    # 4. Extender desde el pico mientras |rho| < epsilon (fechas válidas consecutivas de ene-mar)
    rho = em[columna].pct_change()
    i1 = ip
    while i1 > 0 and abs(rho[i1]) < eps:
        i1 -= 1
    i2 = ip
    while i2 < len(em) - 1 and abs(rho[i2 + 1]) < eps:
        i2 += 1
    t1, t2 = em.loc[i1, "fecha"], em.loc[i2, "fecha"]

    # 5. Aviso si la meseta podría seguir fuera de ene-mar (la fecha válida vecina también es estable)
    aviso = []
    previa = s[s["fecha"] < em["fecha"].iloc[0]]
    posterior = s[s["fecha"] > em["fecha"].iloc[-1]]
    if i1 == 0 and len(previa) and abs(em.loc[0, columna] / previa[columna].iloc[-1] - 1) < eps:
        aviso.append("la meseta podría empezar antes del 1-ene")
    if i2 == len(em) - 1 and len(posterior) and abs(posterior[columna].iloc[0] / em.loc[i2, columna] - 1) < eps:
        aviso.append("la meseta podría seguir después del 31-mar")
    return {**res, "epsilon": round(eps, 4), "t1": t1.date(), "t2": t2.date(),
            "doy_t1": t1.dayofyear, "doy_t2": t2.dayofyear, "dias": (t2 - t1).days,
            "motivo": "ok" + (" (" + "; ".join(aviso) + ")" if aviso else "")}


por_fecha = pd.read_csv("../results/tables/serie_gcc_por_fecha_rio_segundo_2021.csv")
for col in ["agricola", "soja"]:
    print(col, ventana_por_pico(por_fecha, col), "\n")

In [ ]:
import ee
import pandas as pd
ee.Initialize(project="ee-mnolasco")

base = "users/mnolasco/FORMAR_2024_2026/"
deptos = ee.FeatureCollection(base + "departments_selected")
agricola = ee.Image(base + "AgriLand-2019").eq(1)
cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

def preparar_fecha(i):
    refl = i.select(["B2", "B3", "B4"]).divide(10000).toFloat()
    return refl.updateMask(i.select("cs_cdf").gte(0.60)).addBands(i.select("cs_cdf").rename("cs"))

def serie_por_fecha(link, campania=2021):
    """GCC por fecha (área agrícola y soja INTA) para un departamento y una campaña."""
    feat = ee.Feature(deptos.filter(ee.Filter.eq("link", link)).first())
    info = feat.toDictionary(["departamen", "provincia"]).getInfo()
    geom = feat.geometry()
    soja_inta = ee.Image(base + f"MNC_verano-{campania}").eq(11)
    kw = dict(geometry=geom, scale=60, maxPixels=1e13, tileScale=4)

    total = agricola.selfMask().reduceRegion(ee.Reducer.count(), **kw).values().get(0).getInfo()
    S2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(geom)
          .filterDate(f"{campania - 1}-11-01", f"{campania}-06-01").linkCollection(cs, ["cs_cdf"]))
    fechas = sorted(set(S2.aggregate_array("system:time_start")
                          .map(lambda t: ee.Date(t).format("YYYY-MM-dd")).getInfo()))
    reductor = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)

    filas = []
    for k, f in enumerate(fechas, 1):
        d = ee.Date(f)
        dia = S2.filterDate(d, d.advance(1, "day")).map(preparar_fecha).mosaic()
        g = dia.select("B3").divide(dia.select("B2").add(dia.select("B3")).add(dia.select("B4")))
        img = (g.updateMask(agricola).rename("agricola")
                .addBands(g.updateMask(agricola.And(soja_inta)).rename("soja"))
                .addBands(dia.select("cs").updateMask(agricola)))
        r = img.reduceRegion(reductor, **kw).getInfo()
        filas.append({"fecha": f, "cobertura": 100 * (r.get("agricola_count") or 0) / total,
                      "cs_medio": r.get("cs_mean"), "agricola": r.get("agricola_mean"),
                      "soja": r.get("soja_mean")})
        if k % 10 == 0 or k == len(fechas):
            print(f"   {k}/{len(fechas)} fechas")

    df = pd.DataFrame(filas)
    df.to_csv(f"../results/tables/serie_gcc_por_fecha_{link}_{campania}.csv", index=False)
    return df, info

for link in ["06854", "82084"]:
    print(f"\n=== {link} ===")
    df, info = serie_por_fecha(link)
    print(info)
    for col in ["agricola", "soja"]:
        print(col, ventana_por_pico(df, col))

In [ ]:
for link in ["06854", "82084"]:
    df = pd.read_csv(f"../results/tables/serie_gcc_por_fecha_{link}_2021.csv")
    df = df[(df["fecha"] >= "2020-12-15") & (df["fecha"] < "2021-04-15")]
    print(f"\n=== {link} ===")
    print(df.round(3).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

archivos = {"Río Segundo (Córdoba)": "serie_gcc_por_fecha_rio_segundo_2021.csv",
            "25 de Mayo (Buenos Aires)": "serie_gcc_por_fecha_06854_2021.csv",
            "Rosario (Santa Fe)": "serie_gcc_por_fecha_82084_2021.csv"}

fig, axes = plt.subplots(2, 3, figsize=(16, 6), sharex="col",
                         gridspec_kw={"height_ratios": [3, 1]})

for j, (nombre, archivo) in enumerate(archivos.items()):
    df = pd.read_csv("../results/tables/" + archivo)
    df["fecha"] = pd.to_datetime(df["fecha"])
    ok = df["cobertura"] >= 50

    ax = axes[0, j]
    for col, color in [("agricola", "tab:green"), ("soja", "tab:orange")]:
        ax.plot(df.loc[ok, "fecha"], df.loc[ok, col], "o-", color=color, ms=4,
                label=f"{col} (válida)")
        ax.plot(df.loc[~ok, "fecha"], df.loc[~ok, col], "x", color=color, alpha=0.5,
                label=f"{col} (cobertura < 50 %)")
    ax.axvspan(pd.Timestamp("2021-01-01"), pd.Timestamp("2021-03-31"), color="0.92", zorder=0)
    ax.set_title(nombre)
    ax.set_ylabel("GCC")
    ax.set_ylim(0.28, 0.56)
    ax.grid(alpha=0.3)

    axb = axes[1, j]
    axb.bar(df["fecha"], df["cobertura"], width=3,
            color=["tab:blue" if o else "0.7" for o in ok])
    axb.axhline(50, color="red", lw=1, ls="--")
    axb.set_ylabel("Cobertura (%)")
    axb.set_ylim(0, 105)

axes[0, 0].legend(fontsize=7, loc="upper left")
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig("../results/figures/series_gcc_3_departamentos_2021.png", dpi=150)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

archivos = {"Río Segundo (Córdoba)": ("rio_segundo", "serie_gcc_por_fecha_rio_segundo_2021.csv"),
            "25 de Mayo (Buenos Aires)": ("06854", "serie_gcc_por_fecha_06854_2021.csv"),
            "Rosario (Santa Fe)": ("82084", "serie_gcc_por_fecha_82084_2021.csv")}

for nombre, (codigo, archivo) in archivos.items():
    df = pd.read_csv("../results/tables/" + archivo)
    df["fecha"] = pd.to_datetime(df["fecha"])
    ok = df["cobertura"] >= 50

    fig, (ax, axb) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                  gridspec_kw={"height_ratios": [3, 1]})

    for col, color in [("agricola", "tab:green"), ("soja", "tab:orange")]:
        ax.plot(df.loc[ok, "fecha"], df.loc[ok, col], "o-", color=color, ms=5,
                label=f"{col} (válida)")
        ax.plot(df.loc[~ok, "fecha"], df.loc[~ok, col], "x", color=color, alpha=0.5, ms=7,
                label=f"{col} (cobertura < 50 %)")
    ax.axvspan(pd.Timestamp("2021-01-01"), pd.Timestamp("2021-03-31"), color="0.92", zorder=0)
    ax.set_title(nombre, fontsize=14)
    ax.set_ylabel("GCC")
    ax.set_ylim(0.28, 0.56)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9, loc="upper left")

    axb.bar(df["fecha"], df["cobertura"], width=3,
            color=["tab:blue" if o else "0.7" for o in ok])
    axb.axhline(50, color="red", lw=1, ls="--")
    axb.set_ylabel("Cobertura (%)")
    axb.set_ylim(0, 105)
    axb.grid(alpha=0.3)

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.savefig(f"../results/figures/serie_gcc_{codigo}_2021.png", dpi=150)
    plt.show()

In [ ]:
df = pd.read_csv("../results/tables/serie_gcc_por_fecha_06854_2021.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
completa = df["cobertura"] >= 90
parcial = (df["cobertura"] >= 50) & ~completa

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.loc[completa, "fecha"], df.loc[completa, "agricola"], "o-", color="tab:green",
        label="cobertura ≥ 90 %")
ax.plot(df.loc[parcial, "fecha"], df.loc[parcial, "agricola"], "s", color="tab:red",
        label="cobertura 50–90 %")
ax.axvspan(pd.Timestamp("2021-01-01"), pd.Timestamp("2021-03-31"), color="0.92", zorder=0)
ax.set_title("25 de Mayo — GCC área agrícola según cobertura")
ax.set_ylabel("GCC")
ax.set_ylim(0.28, 0.56)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

for cob in [50, 90]:
    print(f"cobertura ≥ {cob} %:", ventana_por_pico(df, "agricola", cobertura_min=cob))

In [ ]:
import ee
import pandas as pd

ee.Initialize(project="ee-mnolasco")

BASE = "users/mnolasco/FORMAR_2024_2026"
deptos = ee.FeatureCollection(f"{BASE}/departments_selected")
agricola = ee.Image(f"{BASE}/AgriLand-2019").eq(1).selfMask()

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(deptos)
      .filterDate("2021-01-01", "2021-03-01"))

orbitas = sorted(s2.aggregate_array("SENSING_ORBIT_NUMBER").distinct().getInfo())
print("Órbitas relativas que tocan la zona:", orbitas)

filas = []
for o in orbitas:
    # huella de la órbita: 1 donde alguna de sus fechas tiene dato
    huella = (s2.filter(ee.Filter.eq("SENSING_ORBIT_NUMBER", o))
                .map(lambda i: i.select("B2").mask().rename("h"))
                .max())
    img = ee.Image.cat(agricola.rename("total"),
                       agricola.updateMask(huella).rename("cubierta"))
    res = img.reduceRegions(collection=deptos.select(["link", "departamen", "provincia"]),
                            reducer=ee.Reducer.count(),
                            crs="EPSG:32720", scale=100, tileScale=4).getInfo()
    for f in res["features"]:
        p = f["properties"]
        filas.append({"link": str(p["link"]).zfill(5), "departamento": p["departamen"],
                      "provincia": p["provincia"], "orbita": o,
                      "total": p.get("total", 0), "cubierta": p.get("cubierta", 0)})
    print(f"órbita {o}: listo")

df = pd.DataFrame(filas)
df["pct"] = 100 * df["cubierta"] / df["total"]
tabla = df.pivot_table(index=["link", "departamento", "provincia"],
                       columns="orbita", values="pct").round(1)
tabla["max_pct"] = tabla[orbitas].max(axis=1)
tabla["n_orbitas"] = (tabla[orbitas] > 1).sum(axis=1)
tabla.to_csv("../results/tables/cobertura_orbitas_departamentos.csv")

completos = (tabla["max_pct"] >= 99).sum()
print(f"\nCubiertos completos por al menos una órbita: {completos} de {len(tabla)}")
print(f"Sin ninguna órbita completa: {len(tabla) - completos}\n")
print(tabla[tabla["max_pct"] < 99].sort_values("max_pct").to_string())

In [ ]:
print(tabla.loc["06854"].to_string())          # control: una órbita ~100 %, otra ~75 %
print("\nDepartamentos según cuántas órbitas los tocan:")
print(tabla["n_orbitas"].value_counts().sort_index())

In [ ]:
import os, time
import ee
import pandas as pd

ee.Initialize(project="ee-mnolasco")

BASE = "users/mnolasco/FORMAR_2024_2026"
CAMPANIA = 2021
SALIDA = f"../results/tables/serie_gcc_95deptos_{CAMPANIA}.csv"
PRUEBA = None          # primero 3 fechas para medir el tiempo; después poner None

deptos = ee.FeatureCollection(f"{BASE}/departments_selected").select(["link"])
agricola = ee.Image(f"{BASE}/AgriLand-2019").eq(1).selfMask()
soja = agricola.updateMask(ee.Image(f"{BASE}/MNC_verano-{CAMPANIA}").eq(11))   # solo control

cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
s2_base = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(deptos)
      .filterDate(f"{CAMPANIA - 1}-11-01", f"{CAMPANIA}-06-01")
      .linkCollection(cs, ["cs_cdf"])
      .map(lambda i: i.set("clave",
           ee.Date(i.get("system:time_start")).format("YYYY-MM-dd").cat("_")
             .cat(ee.Number(i.get("SENSING_ORBIT_NUMBER")).toInt().format("%d")))))

claves = sorted(s2.aggregate_array("clave").distinct().getInfo())
hechas = set(pd.read_csv(SALIDA)["clave"]) if os.path.exists(SALIDA) else set()
pendientes = [c for c in claves if c not in hechas]
print(f"{len(claves)} fecha-órbita en total; faltan {len(pendientes)}")

def gcc(i):
    r = i.select(["B2", "B3", "B4"]).divide(10000).toFloat()
    r = r.updateMask(i.select("cs_cdf").gte(0.60))
    return r.select("B3").divide(r.select("B2").add(r.select("B3")).add(r.select("B4"))).rename("g")

red = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)
lista = pendientes[:PRUEBA] if PRUEBA else pendientes

for n, c in enumerate(lista, 1):
    t0 = time.time()
    fecha, orb = c[:10], int(c[11:])
    col = (s2_base
           .filterDate(fecha, ee.Date(fecha).advance(1, "day"))
           .filter(ee.Filter.eq("SENSING_ORBIT_NUMBER", orb))
           .filterBounds(deptos)
           .linkCollection(cs, ["cs_cdf"]))
    g = col.map(gcc).mosaic()
    img = ee.Image.cat(agricola.rename("total"),
                       g.updateMask(agricola).rename("agr"),
                       g.updateMask(soja).rename("soja"))
    fc = img.reduceRegions(collection=deptos.filterBounds(col.geometry()),
                           reducer=red, crs="EPSG:32720", scale=60, tileScale=4)
    res = None
    for intento in range(3):
        try:
            res = fc.getInfo()
            break
        except Exception as e:
            print(f"  {c}: error ({str(e)[:80]}); reintento en 15 s")
            time.sleep(15)
    if res is None:
        continue
    filas = []
    for f in res["features"]:
        p = f["properties"]
        if not p.get("total_count"):
            continue
        filas.append({"clave": c, "fecha": fecha, "orbita": orb,
                      "link": str(p["link"]).zfill(5),
                      "gcc_agr": p.get("agr_mean"), "gcc_soja": p.get("soja_mean"),
                      "total": p["total_count"], "validos": p.get("agr_count", 0)})
    if filas:
        pd.DataFrame(filas).to_csv(SALIDA, mode="a", header=not os.path.exists(SALIDA), index=False)
    print(f"[{n}/{len(lista)}] {c}: {len(filas)} deptos, {time.time() - t0:.0f} s")

print(f"Pendientes después de esta corrida: {len(pendientes) - len(lista)}")

In [ ]:
import pandas as pd

df = pd.read_csv("../results/tables/serie_gcc_95deptos_2021.csv",
                 dtype={"link": str}, parse_dates=["fecha"])
df["cobertura"] = 100 * df["validos"] / df["total"]

# 1. Control de integridad
print(df["clave"].nunique(), "fecha-órbita;", len(df), "filas;", df["link"].nunique(), "departamentos")
print("Filas duplicadas (clave, link):", df.duplicated(["clave", "link"]).sum())
print("Cobertura máxima:", round(df["cobertura"].max(), 2), "%")

# 2. Fechas útiles por departamento en enero–marzo
em = df[(df["fecha"] >= "2021-01-01") & (df["fecha"] < "2021-04-01")]
links = sorted(df["link"].unique())
n = pd.DataFrame({
    "fechas_ene_mar": em.groupby("link").size(),
    "con_95": em[em["cobertura"] >= 95].groupby("link").size(),
    "con_99": em[em["cobertura"] >= 99].groupby("link").size(),
}).reindex(links).fillna(0).astype(int)

print("\nFechas-órbita por departamento en enero–marzo:")
print(n.describe().round(1).to_string())
print("\nDepartamentos sin ninguna fecha ≥ 95 %:", (n["con_95"] == 0).sum())
print("Departamentos con 1–2 fechas ≥ 95 %:", n["con_95"].between(1, 2).sum())
print("\nLos 10 con menos fechas útiles:")
print(n.sort_values("con_95").head(10).to_string())

In [ ]:
import pandas as pd

cob = pd.read_csv("../results/tables/cobertura_orbitas_departamentos.csv",
                  dtype={"link": str})[["link", "departamento", "provincia"]]

def seleccionar(g, c_min=95, campania=2021):
    g = g.sort_values("fecha")
    em = g[(g["fecha"] >= f"{campania}-01-01") & (g["fecha"] < f"{campania}-04-01")]
    ref = em[em["cobertura"] >= 50]
    pico = ref.loc[ref["gcc_agr"].idxmax()]
    cand = em[em["cobertura"] >= c_min]
    sel = cand.loc[cand["gcc_agr"].idxmax()]
    base = g[g["cobertura"] >= c_min]["gcc_agr"].min()          # mínimo de la campaña nov–may
    return pd.Series({
        "fecha_sel": sel["fecha"].date(), "doy_sel": sel["fecha"].dayofyear,
        "orbita_sel": int(sel["orbita"]), "cob_sel": round(sel["cobertura"], 1),
        "gcc_sel": round(sel["gcc_agr"], 4), "gcc_soja_sel": round(sel["gcc_soja"], 4),
        "fecha_pico": pico["fecha"].date(), "cob_pico": round(pico["cobertura"], 1),
        "gcc_pico": round(pico["gcc_agr"], 4),
        "dias_al_pico": abs((sel["fecha"] - pico["fecha"]).days),
        "gcc_base": round(base, 4),
        "pct_recorrido": round(100 * (sel["gcc_agr"] - base) / (pico["gcc_agr"] - base), 1),
    })

sel = df.groupby("link").apply(seleccionar).reset_index().merge(cob, on="link")
sel.to_csv("../results/tables/seleccion_imagen_2021.csv", index=False)

print("GCC de la imagen elegida (área agrícola):")
print(sel["gcc_sel"].describe().round(4).to_string())
print("\nDepartamentos con GCC elegido < 0.45:", (sel["gcc_sel"] < 0.45).sum())
print("Departamentos con GCC elegido < 0.42:", (sel["gcc_sel"] < 0.42).sum())

print("\nDía del año elegido, por provincia:")
print(sel.groupby("provincia")["doy_sel"].describe()[["count", "min", "50%", "max"]].to_string())

print("\nDías entre la fecha elegida y el pico:")
print(sel["dias_al_pico"].describe().round(1).to_string())
print("\nPorcentaje del recorrido base→pico alcanzado:")
print(sel["pct_recorrido"].describe().round(1).to_string())

In [ ]:
def salto(g, fecha, c_min=95):
    v = g[g["cobertura"] >= c_min].sort_values("fecha").reset_index(drop=True)
    i = v.index[v["fecha"] == pd.Timestamp(fecha)][0]
    vec = [j for j in (i - 1, i + 1) if 0 <= j < len(v)]
    dif = v.loc[i, "gcc_agr"] - v.loc[vec, "gcc_agr"].mean()
    dias = [abs((v.loc[j, "fecha"] - v.loc[i, "fecha"]).days) for j in vec]
    return pd.Series({"salto": round(dif, 4), "dias_vecinas": dias})

sel[["salto", "dias_vecinas"]] = [salto(df[df["link"] == l], f) for l, f in zip(sel["link"], sel["fecha_sel"])]

print("Fechas elegidas más frecuentes:")
print(sel["fecha_sel"].value_counts().head(8).to_string())

print("\nSalto según fecha elegida (las 5 más frecuentes):")
top = sel["fecha_sel"].value_counts().head(5).index
print(sel[sel["fecha_sel"].isin(top)].groupby("fecha_sel")["salto"].describe()[["count", "mean", "min", "max"]].round(4).to_string())

print("\nLos 15 departamentos con mayor salto:")
print(sel.sort_values("salto", ascending=False)[["link", "departamento", "provincia", "fecha_sel",
                                                  "gcc_sel", "salto", "dias_vecinas"]].head(15).to_string(index=False))

In [ ]:
sel["salto"] = sel["salto"].astype(float)

print("Salto según fecha elegida (las 5 más frecuentes):")
top = sel["fecha_sel"].value_counts().head(5).index
print(sel[sel["fecha_sel"].isin(top)].groupby("fecha_sel")["salto"]
      .describe()[["count", "mean", "min", "max"]].round(4).to_string())

print("\nLos 15 departamentos con mayor salto:")
print(sel.sort_values("salto", ascending=False)[["link", "departamento", "provincia", "fecha_sel",
                                                  "gcc_sel", "salto", "dias_vecinas"]].head(15).to_string(index=False))

In [ ]:
import ee
import pandas as pd

ee.Initialize(project="ee-mnolasco")
BASE = "users/mnolasco/FORMAR_2024_2026"
LINK = "82084"   # Rosario

# fecha elegida y sus vecinas válidas (≥ 95 %), con su órbita
v = df[(df["link"] == LINK) & (df["cobertura"] >= 95)].sort_values("fecha").reset_index(drop=True)
i = v.index[v["fecha"] == "2021-02-05"][0]
casos = v.loc[[i - 1, i, i + 1], ["fecha", "orbita"]]

geom = ee.FeatureCollection(f"{BASE}/departments_selected").filter(ee.Filter.eq("link", LINK)).geometry()
agr = ee.Image(f"{BASE}/AgriLand-2019").eq(1).unmask(0)
urbano = ee.ImageCollection("ESA/WorldCover/v200").first().eq(50)
cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

filas = []
for _, c in casos.iterrows():
    f = c["fecha"].strftime("%Y-%m-%d")
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(f, ee.Date(f).advance(1, "day"))
           .filter(ee.Filter.eq("SENSING_ORBIT_NUMBER", int(c["orbita"])))
           .filterBounds(geom).linkCollection(cs, ["cs_cdf"]))
    img = col.map(lambda i: i.select(["B2", "B3", "B4"]).divide(10000).toFloat()
                            .updateMask(i.select("cs_cdf").gte(0.60))).mosaic()
    gcc = img.select("B3").divide(img.select("B2").add(img.select("B3")).add(img.select("B4"))).rename("gcc")
    todo = img.addBands(gcc)
    for zona, mascara in [("agrícola", agr.eq(1)), ("no agrícola", agr.eq(0)), ("urbana", urbano)]:
        r = todo.updateMask(mascara).reduceRegion(ee.Reducer.mean(), geom, scale=60,
                                                  crs="EPSG:32720", maxPixels=1e13, tileScale=4).getInfo()
        filas.append({"fecha": f, "zona": zona, **{k: round(r[k], 4) for k in ["B2", "B3", "B4", "gcc"]}})

res = pd.DataFrame(filas)
print(res.pivot(index="fecha", columns="zona", values=["gcc", "B2", "B3", "B4"]).to_string())

In [ ]:
def seleccionar_robusta(g, c_min=95, campania=2021):
    v = g[g["cobertura"] >= c_min].sort_values("fecha").reset_index(drop=True)
    v["mediana3"] = v["gcc_agr"].rolling(3, center=True).median()
    em = v[(v["fecha"] >= f"{campania}-01-01") & (v["fecha"] < f"{campania}-04-01")]
    k = em["mediana3"].idxmax()
    ventana = v.loc[k - 1:k + 1]                      # las 3 fechas de esa mediana
    e = ventana.loc[(ventana["gcc_agr"] - v.loc[k, "mediana3"]).abs().idxmin()]
    return pd.Series({"fecha_rob": e["fecha"].date(), "orbita_rob": int(e["orbita"]),
                      "cob_rob": round(e["cobertura"], 1), "gcc_rob": round(e["gcc_agr"], 4)})

rob = df.groupby("link").apply(seleccionar_robusta).reset_index()
comp = sel.merge(rob, on="link")
comp["cambio"] = comp["fecha_sel"] != comp["fecha_rob"]
comp["doy_rob"] = pd.to_datetime(comp["fecha_rob"]).dt.dayofyear
comp.to_csv("../results/tables/seleccion_imagen_2021.csv", index=False)

print("Departamentos que cambian de fecha:", comp["cambio"].sum(), "de", len(comp))
print("\nFechas elegidas con la regla robusta:")
print(comp["fecha_rob"].value_counts().head(8).to_string())
print("\nDía del año elegido, por provincia (regla robusta):")
print(comp.groupby("provincia")["doy_rob"].describe()[["count", "min", "50%", "max"]].to_string())
print("\nFechas fuera de enero–marzo:", (~comp["doy_rob"].between(1, 90)).sum())
print("\nCambios (primeros 20):")
print(comp[comp["cambio"]][["link", "departamento", "provincia", "fecha_sel", "gcc_sel",
                            "fecha_rob", "gcc_rob"]].head(20).to_string(index=False))

In [ ]:
import io, urllib.request
import matplotlib.pyplot as plt
from PIL import Image

VIS = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.20, "gamma": 1.2}
borde = (ee.Image().byte()
         .paint(ee.FeatureCollection([ee.Feature(geom)]), 1, 2)
         .visualize(palette=["ffff00"]))
region = geom.bounds().buffer(2000)

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
for ax, (_, c) in zip(axes, casos.iterrows()):
    f = c["fecha"].strftime("%Y-%m-%d")
    img = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(f, ee.Date(f).advance(1, "day"))
           .filter(ee.Filter.eq("SENSING_ORBIT_NUMBER", int(c["orbita"])))
           .filterBounds(geom)
           .select(["B4", "B3", "B2"]).mosaic().divide(10000))
    url = img.visualize(**VIS).blend(borde).getThumbURL(
        {"region": region, "dimensions": 1024, "format": "png"})
    ax.imshow(Image.open(io.BytesIO(urllib.request.urlopen(url).read())))
    gcc = res[(res["fecha"] == f) & (res["zona"] == "agrícola")]["gcc"].iloc[0]
    ax.set_title(f"{f} (órbita {int(c['orbita'])}) — GCC agrícola {gcc:.3f}", fontsize=12)
    ax.axis("off")

fig.suptitle("Rosario (82084) — color real, mismo estiramiento en las tres fechas", fontsize=14)
fig.tight_layout()
fig.savefig("../results/figures/rosario_color_real_5feb2021.png", dpi=150)
plt.show()

In [ ]:
import time
import ee
import pandas as pd

ee.Initialize(project="ee-mnolasco")
BASE = "users/mnolasco/FORMAR_2024_2026"
T = "../results/tables"

# --- 1. Datos y vuelta a la regla del máximo ------------------------------
df = pd.read_csv(f"{T}/serie_gcc_95deptos_2021.csv", dtype={"link": str}, parse_dates=["fecha"])
df["cobertura"] = 100 * df["validos"] / df["total"]
sel = pd.read_csv(f"{T}/seleccion_imagen_2021.csv", dtype={"link": str}, parse_dates=["fecha_sel"])
sel = sel.drop(columns=[c for c in sel.columns if c.endswith("_rob") or c == "cambio"])

# --- 2. Fechas a consultar: elegida + vecina anterior y posterior (≥ 95 %) --
pedidos = []
for _, s in sel.iterrows():
    v = df[(df["link"] == s["link"]) & (df["cobertura"] >= 95)].sort_values("fecha").reset_index(drop=True)
    i = v.index[(v["fecha"] == s["fecha_sel"]) & (v["orbita"] == s["orbita_sel"])][0]
    for rol, j in [("anterior", i - 1), ("elegida", i), ("posterior", i + 1)]:
        if 0 <= j < len(v):
            pedidos.append({"link": s["link"], "rol": rol,
                            "fecha": v.loc[j, "fecha"], "orbita": int(v.loc[j, "orbita"])})
ped = pd.DataFrame(pedidos)
claves = ped[["fecha", "orbita"]].drop_duplicates().reset_index(drop=True)
print(len(ped), "pedidos;", len(claves), "fecha-órbita distintas")

# --- 3. GCC urbano por fecha-órbita ----------------------------------------
deptos = ee.FeatureCollection(f"{BASE}/departments_selected").select(["link"])
urbano = ee.ImageCollection("ESA/WorldCover/v200").first().eq(50).selfMask()
cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
s2_base = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
red = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)

filas = []
for n, k in claves.iterrows():
    t0 = time.time()
    f = k["fecha"].strftime("%Y-%m-%d")
    links = ped[(ped["fecha"] == k["fecha"]) & (ped["orbita"] == k["orbita"])]["link"].unique().tolist()
    fc = deptos.filter(ee.Filter.inList("link", links))
    col = (s2_base.filterDate(f, ee.Date(f).advance(1, "day"))
           .filter(ee.Filter.eq("SENSING_ORBIT_NUMBER", int(k["orbita"])))
           .filterBounds(fc).linkCollection(cs, ["cs_cdf"]))
    img = col.map(lambda i: i.select(["B2", "B3", "B4"]).divide(10000).toFloat()
                            .updateMask(i.select("cs_cdf").gte(0.60))).mosaic()
    gcc = img.select("B3").divide(img.select("B2").add(img.select("B3")).add(img.select("B4"))).rename("gcc")
    u = img.addBands(gcc).updateMask(urbano)
    res = None
    for intento in range(3):
        try:
            res = u.reduceRegions(fc, red, scale=30, crs="EPSG:32720", tileScale=4).getInfo()
            break
        except Exception as e:
            print(f"  {f}: error ({str(e)[:70]}); reintento"); time.sleep(15)
    if res is None:
        continue
    for feat in res["features"]:
        p = feat["properties"]
        filas.append({"link": p["link"], "fecha": k["fecha"], "orbita": int(k["orbita"]),
                      "gcc_urb": p.get("gcc_mean"), "n_urb": p.get("gcc_count", 0)})
    print(f"[{n + 1}/{len(claves)}] {f} órbita {int(k['orbita'])}: {len(links)} deptos, {time.time() - t0:.0f} s")

# --- 4. Salto urbano y guardado --------------------------------------------
urb = ped.merge(pd.DataFrame(filas), on=["link", "fecha", "orbita"], how="left")
tab = urb.pivot_table(index="link", columns="rol", values="gcc_urb")
sel = sel.set_index("link")
sel["salto_urb"] = (tab["elegida"] - tab[["anterior", "posterior"]].mean(axis=1)).round(4)
sel["n_urb"] = urb[urb["rol"] == "elegida"].set_index("link")["n_urb"]
sel = sel.reset_index()
sel.to_csv(f"{T}/seleccion_imagen_2021.csv", index=False)

print("\nControl — Rosario (esperado ≈ +0.008):")
print(sel.loc[sel["link"] == "82084", ["departamento", "fecha_sel", "salto", "salto_urb", "n_urb"]].to_string(index=False))
print("\nSalto urbano en la fecha elegida (95 departamentos):")
print(sel["salto_urb"].describe().round(4).to_string())
print("\nLos 15 con mayor salto urbano (en valor absoluto):")
print(sel.reindex(sel["salto_urb"].abs().sort_values(ascending=False).index)
      [["link", "departamento", "provincia", "fecha_sel", "salto", "salto_urb", "n_urb"]]
      .head(15).to_string(index=False))